##Data Exploration

In [59]:
import pandas as pd

data = pd.read_csv("Sample - Superstore.csv", encoding="latin1")

print(data.shape)
print(data.columns.tolist())
print(data.head())
print(data.info())

(9994, 21)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
1       2  CA-2016-152156   11/8/2016  11/11/2016    Second Class    CG-12520   
2       3  CA-2016-138688   6/12/2016   6/16/2016    Second Class    DV-13045   
3       4  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   
4       5  US-2015-108966  10/11/2015  10/18/2015  Standard Class    SO-20335   

     Customer Name    Segment        Country             City  ...  \
0      Claire Gute   Consumer  United States        Henderson  ...   
1      Claire Gute   Consumer  United States        Henderson  ...   
2  Darrin Van Huff  Corporate  U

##Data Profiling

In [60]:
print("Unique Orders:", data["Order ID"].nunique())

print("Unique Customers:", data["Customer ID"].nunique())

print("Unique Products:", data["Product ID"].nunique())

print("Unique Cities:", data["City"].nunique())

print("Unique States:", data["State"].nunique())

print("Unique Regions:", data["Region"].nunique())

print("Unique Postal Codes:", data["Postal Code"].nunique())

Unique Orders: 5009
Unique Customers: 793
Unique Products: 1862
Unique Cities: 531
Unique States: 49
Unique Regions: 4
Unique Postal Codes: 631


##Database Design

In [61]:
# Convert dates
data["Order Date"] = pd.to_datetime(data["Order Date"])
data["Ship Date"] = pd.to_datetime(data["Ship Date"])

# Verify
print(data.dtypes)

Row ID                    int64
Order ID                 object
Order Date       datetime64[ns]
Ship Date        datetime64[ns]
Ship Mode                object
Customer ID              object
Customer Name            object
Segment                  object
Country                  object
City                     object
State                    object
Postal Code               int64
Region                   object
Product ID               object
Category                 object
Sub-Category             object
Product Name             object
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object


###**Customer table**

In [62]:
customers = (
    data[["Customer ID", "Customer Name", "Segment"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

customers.head()

,Customer ID,Customer Name,Segment
0,CG-12520,Claire Gute,Consumer
1,DV-13045,Darrin Van Huff,Corporate
2,SO-20335,Sean O'Donnell,Consumer
3,BH-11710,Brosina Hoffman,Consumer
4,AA-10480,Andrew Allen,Consumer


In [63]:
print(customers.shape)
print(customers.head())

(793, 3)
  Customer ID    Customer Name    Segment
0    CG-12520      Claire Gute   Consumer
1    DV-13045  Darrin Van Huff  Corporate
2    SO-20335   Sean O'Donnell   Consumer
3    BH-11710  Brosina Hoffman   Consumer
4    AA-10480     Andrew Allen   Consumer


###**Products table**

In [64]:
products = (
    data[["Product ID", "Category", "Sub-Category", "Product Name"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(products.shape)
products.head()

(1894, 4)


,Product ID,Category,Sub-Category,Product Name
0,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase
1,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,..."
2,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...
3,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table
4,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System


In [65]:
products['Product ID'].nunique()

1862

In [66]:
products = (
    data[
        [
            "Product ID",
            "Product Name",
            "Category",
            "Sub-Category"
        ]
    ]
    .drop_duplicates(subset=["Product ID"], keep="first")
    .reset_index(drop=True)
)

print(products.shape)
print(products["Product ID"].duplicated().sum())

(1862, 4)
0


In [67]:
#data validation
locations_check = (
    data.groupby("Postal Code")
    .agg({
        "City": "nunique",
        "State": "nunique",
        "Region": "nunique",
        "Country": "nunique"
    })
)

locations_check.sort_values(
    ["City", "State", "Region"],
    ascending=False
).head(20)

,City,State,Region,Country
Postal Code,,,,
92024,2,1,1,1
1040,1,1,1,1
1453,1,1,1,1
1752,1,1,1,1
1810,1,1,1,1
1841,1,1,1,1
1852,1,1,1,1
1915,1,1,1,1
2038,1,1,1,1


###**Location table**

In [68]:
locations = (
    data[
        [
            "Postal Code",
            "City",
            "State",
            "Region",
            "Country"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(locations.shape)
locations.head()

(632, 5)


,Postal Code,City,State,Region,Country
0,42420,Henderson,Kentucky,South,United States
1,90036,Los Angeles,California,West,United States
2,33311,Fort Lauderdale,Florida,South,United States
3,90032,Los Angeles,California,West,United States
4,28027,Concord,North Carolina,South,United States


In [69]:
locations.insert(
    0,
    "Location_Key",
    range(1, len(locations) + 1)
)

locations.head()

,Location_Key,Postal Code,City,State,Region,Country
0,1,42420,Henderson,Kentucky,South,United States
1,2,90036,Los Angeles,California,West,United States
2,3,33311,Fort Lauderdale,Florida,South,United States
3,4,90032,Los Angeles,California,West,United States
4,5,28027,Concord,North Carolina,South,United States


###**Orders table**

In [70]:
orders = (
    data[
        [
            "Order ID",
            "Order Date",
            "Ship Date",
            "Ship Mode",
            "Customer ID",
            "Postal Code",
            "City",
            "State",
            "Region",
            "Country"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [71]:
orders = orders.merge(
    locations,
    on=[
        "Postal Code",
        "City",
        "State",
        "Region",
        "Country"
    ],
    how="left"
)

In [72]:
orders = orders[
    [
        "Order ID",
        "Order Date",
        "Ship Date",
        "Ship Mode",
        "Customer ID",
        "Location_Key"
    ]
]

In [73]:
print(orders["Location_Key"].isnull().sum())

0


###**Order_details**

In [74]:
order_details = (
    data[
        [
            "Order ID",
            "Product ID",
            "Quantity",
            "Sales",
            "Discount",
            "Profit"
        ]
    ]
    .copy()
)

print(order_details.shape)
order_details.head()

(9994, 6)


,Order ID,Product ID,Quantity,Sales,Discount,Profit
0,CA-2016-152156,FUR-BO-10001798,2,261.9600,0.00,41.9136
1,CA-2016-152156,FUR-CH-10000454,3,731.9400,0.00,219.5820
2,CA-2016-138688,OFF-LA-10000240,2,14.6200,0.00,6.8714
3,US-2015-108966,FUR-TA-10000577,5,957.5775,0.45,-383.0310
4,US-2015-108966,OFF-ST-10000760,2,22.3680,0.20,2.5164


In [75]:
order_details.isnull().sum()

,0
Order ID,0
Product ID,0
Quantity,0
Sales,0
Discount,0
Profit,0


In [76]:
order_details.insert(
    0,
    "Order_Detail_ID",
    range(1, len(order_details) + 1)
)

order_details.head()

,Order_Detail_ID,Order ID,Product ID,Quantity,Sales,Discount,Profit
0,1,CA-2016-152156,FUR-BO-10001798,2,261.9600,0.00,41.9136
1,2,CA-2016-152156,FUR-CH-10000454,3,731.9400,0.00,219.5820
2,3,CA-2016-138688,OFF-LA-10000240,2,14.6200,0.00,6.8714
3,4,US-2015-108966,FUR-TA-10000577,5,957.5775,0.45,-383.0310
4,5,US-2015-108966,OFF-ST-10000760,2,22.3680,0.20,2.5164


In [77]:
# Inspect the completely duplicated transaction
order_details[
    order_details.duplicated(keep=False)
]

,Order_Detail_ID,Order ID,Product ID,Quantity,Sales,Discount,Profit


In [78]:
# Inspect repeated (Order ID, Product ID) combinations
order_details[
    order_details.duplicated(
        subset=["Order ID", "Product ID"],
        keep=False
    )
].sort_values(["Order ID", "Product ID"])

,Order_Detail_ID,Order ID,Product ID,Quantity,Sales,Discount,Profit
6498,6499,CA-2015-103135,OFF-BI-10000069,9,135.090,0.0,62.1414
6500,6501,CA-2015-103135,OFF-BI-10000069,6,90.060,0.0,41.4276
350,351,CA-2016-129714,OFF-PA-10001970,2,24.560,0.0,11.5432
352,353,CA-2016-129714,OFF-PA-10001970,4,49.120,0.0,23.0864
1300,1301,CA-2016-137043,FUR-FU-10003664,6,572.760,0.0,166.1004
1301,1302,CA-2016-137043,FUR-FU-10003664,3,286.380,0.0,83.0502
9168,9169,CA-2016-140571,OFF-PA-10001954,14,319.760,0.0,147.0896
9169,9170,CA-2016-140571,OFF-PA-10001954,2,45.680,0.0,21.0128
7881,7882,CA-2017-118017,TEC-AC-10002006,6,76.752,0.2,10.5534
7882,7883,CA-2017-118017,TEC-AC-10002006,8,102.336,0.2,14.0712


In [79]:
import sqlite3

In [80]:
conn = sqlite3.connect("superstore.db")

cursor = conn.cursor()

In [81]:
cursor.execute("PRAGMA foreign_keys = ON;")

In [82]:
cursor.execute("SELECT sqlite_version();")

print(cursor.fetchone())

('3.37.2',)


In [83]:
conn.commit()

###**Database Engineering**

####**Customer table**

In [84]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Customers (

    Customer_ID TEXT PRIMARY KEY,

    Customer_Name TEXT NOT NULL,

    Segment TEXT NOT NULL

);
""")

In [85]:
conn.commit()

In [86]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('Customers',), ('Products',), ('Locations',), ('Orders',), ('Order_Details',)]


####**Products table**

In [87]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS Products (

    Product_ID TEXT PRIMARY KEY,

    Product_Name TEXT NOT NULL,

    Category TEXT NOT NULL,

    Sub_Category TEXT NOT NULL

);
""")

In [88]:
conn.commit()

####**Location table**

In [89]:
cursor.execute("""
CREATE TABLE Locations (

    Location_Key INTEGER PRIMARY KEY,

    Postal_Code INTEGER NOT NULL,

    City TEXT NOT NULL,

    State TEXT NOT NULL,

    Region TEXT NOT NULL,

    Country TEXT NOT NULL

);
""")
conn.commit()

OperationalError: table Locations already exists

####**Orders table**

In [90]:
cursor.execute("""
CREATE TABLE Orders (

    Order_ID TEXT PRIMARY KEY,

    Order_Date DATE NOT NULL,

    Ship_Date DATE NOT NULL,

    Ship_Mode TEXT NOT NULL,

    Customer_ID TEXT NOT NULL,

    Location_Key INTEGER NOT NULL,

    FOREIGN KEY (Customer_ID)
        REFERENCES Customers(Customer_ID),

    FOREIGN KEY (Location_Key)
        REFERENCES Locations(Location_Key)

);
""")
conn.commit()

OperationalError: table Orders already exists

####**Order_Details**

In [91]:
cursor.execute("""
CREATE TABLE Order_Details (

    Order_Detail_ID INTEGER PRIMARY KEY,

    Order_ID TEXT NOT NULL,

    Product_ID TEXT NOT NULL,

    Quantity INTEGER NOT NULL,

    Sales REAL NOT NULL,

    Discount REAL NOT NULL,

    Profit REAL NOT NULL,

    FOREIGN KEY (Order_ID)
        REFERENCES Orders(Order_ID),

    FOREIGN KEY (Product_ID)
        REFERENCES Products(Product_ID)

);
""")
conn.commit()

OperationalError: table Order_Details already exists

In [92]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('Customers',), ('Products',), ('Locations',), ('Orders',), ('Order_Details',)]


###**Loading Data**

In [93]:
customers = customers.rename(columns={
    "Customer ID": "Customer_ID",
    "Customer Name": "Customer_Name"
})

customers.to_sql(
    "Customers",
    conn,
    if_exists="append",
    index=False
)

IntegrityError: UNIQUE constraint failed: Customers.Customer_ID

In [94]:
products = products.rename(columns={
    "Product ID": "Product_ID",
    "Product Name": "Product_Name",
    "Sub-Category": "Sub_Category"
})

products.to_sql(
    "Products",
    conn,
    if_exists="append",
    index=False
)

IntegrityError: UNIQUE constraint failed: Products.Product_ID

In [95]:
locations.rename(
    columns={"Postal Code": "Postal_Code"},
    inplace=True
)

locations.to_sql(
    "Locations",
    conn,
    if_exists="append",
    index=False
)

IntegrityError: UNIQUE constraint failed: Locations.Location_Key

In [96]:
orders = orders.rename(columns={
    "Order ID": "Order_ID",
    "Order Date": "Order_Date",
    "Ship Date": "Ship_Date",
    "Ship Mode": "Ship_Mode",
    "Customer ID": "Customer_ID"
})

orders.to_sql(
    "Orders",
    conn,
    if_exists="append",
    index=False
)

IntegrityError: UNIQUE constraint failed: Orders.Order_ID

In [97]:
order_details = order_details.rename(columns={
    "Order ID": "Order_ID",
    "Product ID": "Product_ID"
})

order_details.to_sql(
    "Order_Details",
    conn,
    if_exists="append",
    index=False
)

IntegrityError: UNIQUE constraint failed: Order_Details.Order_Detail_ID

In [98]:
conn.commit()

In [99]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('Customers',), ('Products',), ('Locations',), ('Orders',), ('Order_Details',)]


###**Validation**

In [100]:
cursor.execute("SELECT COUNT(*) FROM Customers")
print(cursor.fetchone())

(793,)


In [101]:
cursor.execute("SELECT COUNT(*) FROM Products")
print(cursor.fetchone())

(1862,)


In [102]:
cursor.execute("SELECT COUNT(*) FROM Locations")
print(cursor.fetchone())

(632,)


In [103]:
cursor.execute("SELECT COUNT(*) FROM Orders")
print(cursor.fetchone())

(5009,)


In [104]:
cursor.execute("SELECT COUNT(*) FROM Order_Details")
print(cursor.fetchone())

(9994,)


##**SQL Business Analysis**

####**Executive KPI Dashboard**

In [105]:
query = ('''
SELECT
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    SUM(Quantity) AS Total_Quantity_Sold,
    COUNT(DISTINCT Order_ID) AS Total_Orders,
    ROUND((SUM(Profit) / SUM(Sales)) * 100, 2) AS Profit_Margin_Percent
FROM Order_Details;
''')
data = pd.read_sql_query(query, conn)
data

,Total_Sales,Total_Profit,Total_Quantity_Sold,Total_Orders,Profit_Margin_Percent
0,2297200.86,286397.02,37873,5009,12.47


####**Business Insight**
"The company generated 2.30 US Dollar million in sales from 5,009 customer orders, selling 37,873 units. Overall profitability was $286.4 thousand, resulting in a 12.47% profit margin. While revenue is strong, the relatively modest margin suggests there may be opportunities to improve pricing, reduce discounts, or optimize operating costs."

####**Regional Performance Analysis**

In [106]:
query = ('''
SELECT
    l.Region,
    ROUND(SUM(od.Sales), 2) AS Total_Sales,
    ROUND(SUM(od.Profit), 2) AS Total_Profit,
    SUM(od.Quantity) AS Total_Quantity,
    COUNT(DISTINCT o.Order_ID) AS Total_Orders,
    ROUND((SUM(od.Profit) / SUM(od.Sales)) * 100, 2) AS Profit_Margin
FROM Order_Details od
JOIN Orders o
    ON od.Order_ID = o.Order_ID
JOIN Locations l
    ON o.Location_Key = l.Location_Key
GROUP BY l.Region
ORDER BY Total_Sales DESC;
''')
data = pd.read_sql_query(query, conn)
data

,Region,Total_Sales,Total_Profit,Total_Quantity,Total_Orders,Profit_Margin
0,West,725457.82,108418.45,12266,1611,14.94
1,East,678781.24,91522.78,10618,1401,13.48
2,Central,501239.89,39706.36,8780,1175,7.92
3,South,391721.91,46749.43,6209,822,11.93


####**Business Insights**

The West region is the company's strongest-performing market, leading in sales, profit, order volume, and profit margin. The East region also performs well with healthy profitability. The Central region deserves further investigation because, despite generating over half a million dollars in sales, its profit margin is significantly lower than the other regions. This suggests opportunities to improve pricing, discount policies, product mix, or operational efficiency

####**Product Performance Analysis**

####**Top 10 Most Profitable Products**

In [107]:
query = ('''
SELECT p.Product_Name,
       p.Category,
       ROUND(SUM(od.Sales), 2) AS Total_Sales,
       ROUND(SUM(od.Profit), 2) AS Total_Profit,
       SUM(od.Quantity) AS Total_Quantity_Sold
 FROM Products p
 JOIN Order_Details od
    ON p.Product_ID = od.Product_ID
 GROUP BY p.Product_Name,p.Category
 ORDER BY Total_Profit DESC
 LIMIT 10;

''')
data = pd.read_sql_query(query, conn)
data

,Product_Name,Category,Total_Sales,Total_Profit,Total_Quantity_Sold
0,Canon imageCLASS 2200 Advanced Copier,Technology,61599.82,25199.93,20
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,27453.38,7753.04,31
2,Hewlett Packard LaserJet 3310 Copier,Technology,18839.69,6983.88,38
3,Canon PC1060 Personal Laser Copier,Technology,11619.83,4570.93,19
4,Logitech G19 Programmable Gaming Keyboard,Technology,13756.54,4425.34,60
5,HP Designjet T520 Inkjet Large Format Printer ...,Technology,18374.90,4094.98,12
6,Ativa V4110MDD Micro-Cut Shredder,Technology,7699.89,3772.95,11
7,"3D Systems Cube Printer, 2nd Generation, Magenta",Technology,14299.89,3717.97,11
8,Ibico EPK-21 Electric Binding System,Office Supplies,15875.92,3345.28,13
9,Zebra ZM400 Thermal Label Printer,Technology,6965.70,3343.54,6


#### **Customer Lifetime Value (CLV)**

In [108]:
query = ('''
SELECT c.Customer_Name,
       COUNT(DISTINCT o.Order_ID) AS Total_Orders,
       ROUND(SUM(od.Sales), 2) AS Total_Sales,
       ROUND(SUM(od.Profit), 2) AS Total_Profit,
       ROUND(SUM(od.Sales) / COUNT(DISTINCT o.Order_ID), 2) AS Average_Order_Value,
       RANK() OVER (ORDER BY SUM(od.Profit) DESC) AS Customer_Rank
FROM Customers c
JOIN Orders o
    ON c.Customer_ID = o.Customer_ID
JOIN Order_Details od
    ON o.Order_ID = od.Order_ID
GROUP BY c.Customer_ID,c.Customer_Name
ORDER BY Customer_Rank
''');
data = pd.read_sql_query(query, conn)
data


,Customer_Name,Total_Orders,Total_Sales,Total_Profit,Average_Order_Value,Customer_Rank
0,Tamara Chand,5,19052.22,8981.32,3810.44,1
1,Raymond Buch,6,15117.34,6976.10,2519.56,2
2,Sanjit Chand,9,14142.33,5757.41,1571.37,3
3,Hunter Lopez,6,12873.30,5622.43,2145.55,4
4,Adrian Barton,10,14473.57,5444.81,1447.36,5
...,...,...,...,...,...,...
788,Henry Goldwyn,12,3247.64,-2797.96,270.64,789
789,Sharelle Roach,5,3233.48,-3333.91,646.70,790
790,Luke Foster,7,3930.51,-3583.98,561.50,791
791,Grant Thornton,3,9351.21,-4108.66,3117.07,792


###**Discount Analysis**

In [109]:
query = ('''
SELECT
    Discount,
    COUNT(Order_Detail_ID) AS Number_of_Order_Lines,
    ROUND(SUM(Sales), 2) AS Total_Sales,
    ROUND(SUM(Profit), 2) AS Total_Profit,
    ROUND(SUM(Profit) / COUNT(Order_Detail_ID), 2) AS Average_Profit_Per_Order_Line,
    ROUND(SUM(Profit) / SUM(Sales) * 100, 2) AS Profit_Margin_Percent
FROM Order_Details
GROUP BY Discount
ORDER BY Discount DESC;
''')
data = pd.read_sql_query(query, conn)
display(data)

,Discount,Number_of_Order_Lines,Total_Sales,Total_Profit,Average_Profit_Per_Order_Line,Profit_Margin_Percent
0,0.80,300,16963.76,-30539.04,-101.80,-180.03
1,0.70,418,40620.28,-40075.36,-95.87,-98.66
2,0.60,138,6644.70,-5944.66,-43.08,-89.46
3,0.50,66,58918.54,-20506.43,-310.70,-34.80
4,0.45,11,5484.97,-2493.11,-226.65,-45.45
5,0.40,206,116417.78,-23057.05,-111.93,-19.81
6,0.32,27,14493.46,-2391.14,-88.56,-16.50
7,0.30,227,103226.65,-10369.28,-45.68,-10.05
8,0.20,3657,764594.37,90337.31,24.70,11.82
9,0.15,52,27558.52,1418.99,27.29,5.15


###**Executive Summary**

The report shows a very strong relationship between increasing discounts and declining profitability.

As discount levels increase, profit margins consistently decrease, eventually becoming negative.

This suggests that the company's current discount strategy is reducing, and in many cases eliminating, profitability.


**Insight 1: No Discount (0%)**

Metric	Value
Sales	1,087,908.47
Profit	320,987.60
Margin	29.51%

This is the healthiest pricing level.

Nearly 30 cents of every sales dollar becomes profit.

This serves as our benchmark.


**Insight 2: 10% Discount**

Metric	Value
Margin	16.61%

The company is still profitable.

Although margin drops from 29.51% → 16.61%, the business continues to generate healthy returns.

This suggests that modest discounts can be sustainable.

**Insight 3: 20% Discount**

Metric	Value
Sales	764,594
Profit	90,337
Margin	11.82%

This is interesting.

Notice:

The second-highest sales volume.
Still profitable.
Much lower margin.

This could indicate that 20% discounts drive sales volume, but management should verify whether the increase in sales is worth the reduction in margin.


**Insight 4: The Turning Point**

At 30% discount:

Metric	Value
Profit	-10,369
Margin	-10.05%

This is the first discount level where the business is losing money overall.

From here onward:

32%
40%
45%
50%
60%
70%
80%

all have negative profit margins.

This is a major finding.

Insight 5: The Highest Discounts Are Destructive

Look at 80% discount:

Metric	Value
Sales	16,963.76
Profit	-30,539.04
Margin	-180.03%



###**Recommendation**

1. **Review Discounts Above 20%**

Discounts of 30% and above consistently produce negative profit margins.

These should be reviewed to determine whether they are justified by strategic objectives (e.g., clearing obsolete inventory) or whether they are unnecessarily eroding profitability.

2. **Investigate the 20% Discount Band**

The 20% discount level still generates positive profit, but the margin is substantially lower than at 0% or 10%.

Management should assess whether the increase in sales volume compensates for the reduced margin.

3. **Audit Extreme Discounts (70–80%)**

Discounts of 70% and 80% result in severe financial losses.

Possible explanations include:

Clearance sales.
Pricing errors.
Damaged inventory.
Data quality issues.



####**Performance of each product categor**y

In [110]:
query = ('''
SELECT p.category,
       ROUND(SUM(od.Sales), 2) AS Total_Sales,
       ROUND(SUM(od.Profit), 2) AS Total_Profit,
       SUM(od.Quantity) AS Total_Quantity_Sold,
       COUNT(DISTINCT od.Order_ID) AS Total_Orders,
       ROUND((SUM(od.Profit) / SUM(od.Sales)) * 100, 2) AS Profit_Margin_Percent,
       ROUND(AVG(od.Discount), 2) AS Average_Discount
FROM Products p
JOIN Order_Details od
    ON p.Product_ID = od.Product_ID
GROUP BY p.Category
ORDER BY Total_Profit DESC
''');
data = pd.read_sql_query(query, conn)
data

,Category,Total_Sales,Total_Profit,Total_Quantity_Sold,Total_Orders,Profit_Margin_Percent,Average_Discount
0,Technology,836154.03,145454.95,6939,1544,17.40,0.13
1,Office Supplies,719047.03,122490.80,22906,3742,17.04,0.16
2,Furniture,741999.80,18451.27,8028,1764,2.49,0.17


##**Advance SQL Analytics Series**


###**Customer Segmentation**

In [111]:
query = ('''
WITH CustomerMetrics AS (
    SELECT
        c.Customer_ID,
        c.Customer_Name,
        COUNT(DISTINCT o.Order_ID) AS Total_Orders,
        ROUND(SUM(od.Sales), 2) AS Total_Sales,
        ROUND(SUM(od.Profit), 2) AS Total_Profit,
        ROUND(
            SUM(od.Sales) /
            COUNT(DISTINCT o.Order_ID),
            2
        ) AS Average_Order_Value
    FROM Customers c
    JOIN Orders o
        ON c.Customer_ID = o.Customer_ID
    JOIN Order_Details od
        ON o.Order_ID = od.Order_ID
    GROUP BY
        c.Customer_ID,
        c.Customer_Name
),

RankedCustomers AS (
    SELECT
        *,
        NTILE(10) OVER (
            ORDER BY Total_Profit DESC
        ) AS Profit_Decile
    FROM CustomerMetrics
)

SELECT
    Customer_Name,
    Total_Orders,
    Total_Sales,
    Total_Profit,
    CASE
        WHEN Profit_Decile <= 2 THEN 'VIP'
        WHEN Profit_Decile <= 7 THEN 'Regular'
        ELSE 'Low Value'
    END AS Customer_Segment
FROM RankedCustomers
ORDER BY Total_Profit DESC ''');
data = pd.read_sql_query(query, conn)
data

,Customer_Name,Total_Orders,Total_Sales,Total_Profit,Customer_Segment
0,Tamara Chand,5,19052.22,8981.32,VIP
1,Raymond Buch,6,15117.34,6976.10,VIP
2,Sanjit Chand,9,14142.33,5757.41,VIP
3,Hunter Lopez,6,12873.30,5622.43,VIP
4,Adrian Barton,10,14473.57,5444.81,VIP
...,...,...,...,...,...
788,Henry Goldwyn,12,3247.64,-2797.96,Low Value
789,Sharelle Roach,5,3233.48,-3333.91,Low Value
790,Luke Foster,7,3930.51,-3583.98,Low Value
791,Grant Thornton,3,9351.21,-4108.66,Low Value


###**Pareto Analysis**

In [112]:
query = ('''
WITH CustomerMetrics AS (
SELECT c.Customer_ID,
       c.Customer_Name,
       ROUND(SUM(od.Profit),2) AS Total_Profit

FROM Customers c
JOIN Orders o
    ON c.Customer_ID = o.Customer_ID
JOIN Order_Details od
    ON o.Order_ID = od.Order_ID
GROUP BY c.Customer_ID,
         c.Customer_Name
),
ParetoMetrics AS (
SELECT Customer_ID,
       Customer_Name,
       Total_Profit,
       ROUND(
           Total_Profit * 100.0/
           SUM(Total_Profit) OVER (),2) AS Profit_Contribution_Percentage,
       ROUND(
           SUM(Total_Profit) OVER (ORDER BY Total_Profit DESC), 2) AS Running_Profit,
       ROUND(
           SUM(Total_Profit) OVER (ORDER BY Total_Profit DESC) * 100.0/
           SUM(Total_Profit) OVER (), 2) AS Cumulative_Profit_Percent
FROM CustomerMetrics

),
ParetoAnalysis AS (
SELECT *,
       CASE
           WHEN Cumulative_Profit_Percent <= 80 THEN 'Top 80%'
           ELSE 'Remaining'
      END AS Pareto_Group
FROM ParetoMetrics
)
SELECT *
FROM ParetoAnalysis
ORDER BY Total_Profit DESC;
''')
data = pd.read_sql_query(query, conn)
display(data)

,Customer_ID,Customer_Name,Total_Profit,Profit_Contribution_Percentage,Running_Profit,Cumulative_Profit_Percent,Pareto_Group
0,TC-20980,Tamara Chand,8981.32,3.14,8981.32,3.14,Top 80%
1,RB-19360,Raymond Buch,6976.10,2.44,15957.42,5.57,Top 80%
2,SC-20095,Sanjit Chand,5757.41,2.01,21714.83,7.58,Top 80%
3,HL-15040,Hunter Lopez,5622.43,1.96,27337.26,9.55,Top 80%
4,AB-10105,Adrian Barton,5444.81,1.90,32782.07,11.45,Top 80%
...,...,...,...,...,...,...,...
788,HG-14965,Henry Goldwyn,-2797.96,-0.98,304050.12,106.16,Remaining
789,SR-20425,Sharelle Roach,-3333.91,-1.16,300716.21,105.00,Remaining
790,LF-17185,Luke Foster,-3583.98,-1.25,297132.23,103.75,Remaining
791,GT-14635,Grant Thornton,-4108.66,-1.43,293023.57,102.31,Remaining


**Executive Summary**

**Pareto Analysis Objective **

The objective of this analysis was to determine whether the company's profits follow the Pareto Principle (80/20 Rule), which states that a relatively small proportion of customers often generates the majority of business profit.

**Key Findings**
1. **Customer Profit Distribution**
The company has 793 unique customers.
Customer profitability is highly uneven, with a relatively small group contributing a significant portion of total profit.
2. **Top Customer Contribution**
The most profitable customer generated $8,981.32, representing 3.14% of the company's total profit.
This indicates that the business is not heavily dependent on a single customer, reducing the financial risk associated with losing one account.
3. **Negative Profit Customers**

The analysis identified several customers with negative total profit.

Examples include:

Cindy Stewart
Grant Thornton
Luke Foster
Sharelle Roach
Henry Goldwyn

These customers reduced the company's overall profitability despite generating sales.

Possible business causes include:

High discount levels.
Low-margin products.
Expensive shipping costs.
Inefficient pricing strategies.

These customers should be investigated further before future marketing investments.

4. **Profit Concentration**

The Pareto analysis successfully classified customers into:

Top 80% Profit Contributors
Remaining Customers

This segmentation allows management to identify the customers that contribute most to company profitability.

**Business Recommendations**

**Recommendation 1: Protect High-Value Customers**

Develop a structured loyalty program for customers in the Top 80% Profit segment.

Potential initiatives include:

Priority customer support.
Exclusive promotions.
Dedicated account management.
Early access to new products.

Retaining these customers is likely to have the greatest impact on company profitability.

**Recommendation 2: Investigate Loss-Making Customers**

Conduct a detailed analysis of customers generating negative profit to determine the underlying causes.

Areas for investigation include:

Excessive discounts.
High shipping or fulfillment costs.
Product return rates.
Pricing strategy.

Corrective actions could significantly improve overall profitability.

**Recommendation 3: Introduce Tiered Customer Rewards**

Instead of rewarding all customers equally, implement a tiered loyalty program based on customer value.

Example:

Platinum: Top 50 customers.
Gold: Remaining Top 80% customers.
Silver: Growth-potential customers.

This approach balances customer retention with budget efficiency.

**Recommendation 4: Monitor Customer Profitability Regularly**

Rather than focusing solely on sales, management should monitor customer profitability as a key performance indicator.

Regular profitability reviews can help identify:

Emerging VIP customers.
Declining customer performance.
Loss-making accounts requiring intervention.

###**Time Series Analytics**

####**Monthly Summary**

In [113]:
query = ('''
SELECT
strftime('%Y', o.Order_Date) AS Year,
strftime('%m', o.Order_Date) AS Month,
ROUND(SUM(od.Sales), 2) AS Total_Sales,
ROUND(SUM(od.Profit), 2) AS Total_Profit,
COUNT(DISTINCT o.Order_ID) AS Total_Orders
FROM Orders o
JOIN Order_Details od
    ON o.Order_ID = od.Order_ID
GROUP BY Year, Month
ORDER BY Year , Month;
''')
data = pd.read_sql_query(query, conn)
data

,Year,Month,Total_Sales,Total_Profit,Total_Orders
0,2014,01,14236.90,2450.19,32
1,2014,02,4519.89,862.31,28
2,2014,03,55691.01,498.73,71
3,2014,04,28295.35,3488.84,66
4,2014,05,23648.29,2738.71,69
5,2014,06,34595.13,4976.52,66
6,2014,07,33946.39,-841.48,65
7,2014,08,27909.47,5318.11,72
8,2014,09,81777.35,8328.10,130
9,2014,10,31453.39,3448.26,78


###**Month-Over-Month(MoM) Analysis**

In [114]:
query =('''
WITH MonthlySales AS (
SELECT
strftime('%Y', o.Order_Date) AS Year,
strftime('%m', o.Order_Date) AS Month,
ROUND(SUM(od.Sales), 2) AS Total_Sales,
ROUND(SUM(od.Profit), 2) AS Total_Profit,
COUNT(DISTINCT o.Order_ID) AS Total_Orders
FROM Orders o
JOIN Order_Details od
    ON o.Order_ID = od.Order_ID
GROUP BY Year, Month
ORDER BY Year, Month
),
MonthlyMetrics AS (
SELECT *,
LAG(Total_Sales) OVER (ORDER BY Year, Month) AS Previous_Month_Sales
FROM MonthlySales
)
SELECT*,
ROUND(
    (Total_Sales - Previous_Month_Sales) * 100.0 /
    NULLIF(Previous_Month_Sales,0),
    2
) AS MoM_Sales_Growth_Percent
FROM MonthlyMetrics
ORDER BY Year, Month
''')
data = pd.read_sql_query(query, conn)
data

,Year,Month,Total_Sales,Total_Profit,Total_Orders,Previous_Month_Sales,MoM_Sales_Growth_Percent
0,2014,01,14236.90,2450.19,32,NaN,NaN
1,2014,02,4519.89,862.31,28,14236.90,-68.25
2,2014,03,55691.01,498.73,71,4519.89,1132.13
3,2014,04,28295.35,3488.84,66,55691.01,-49.19
4,2014,05,23648.29,2738.71,69,28295.35,-16.42
5,2014,06,34595.13,4976.52,66,23648.29,46.29
6,2014,07,33946.39,-841.48,65,34595.13,-1.88
7,2014,08,27909.47,5318.11,72,33946.39,-17.78
8,2014,09,81777.35,8328.10,130,27909.47,193.01
9,2014,10,31453.39,3448.26,78,81777.35,-61.54


###**Year-Over-Year(YoY) Analysis**

In [115]:
query =('''
WITH YearlySales AS (
SELECT
strftime('%Y', o.Order_Date) AS Year,
ROUND(SUM(od.Sales), 2) AS Total_Sales,
ROUND(SUM(od.Profit), 2) AS Total_Profit,
COUNT(DISTINCT o.Order_ID) AS Total_Orders
FROM Orders o
JOIN Order_Details od
    ON o.Order_ID = od.Order_ID
GROUP BY Year
ORDER BY Year
),
YearlyMetrics AS (
SELECT *,
LAG(Total_Sales) OVER (ORDER BY Year) AS Previous_Year_Sales
FROM YearlySales
)
SELECT *,
ROUND(
    (Total_Sales - Previous_Year_Sales) * 100.0 /
    NULLIF(Previous_Year_Sales, 0),
    2
) AS YoY_Sales_Growth_Percent
FROM YearlyMetrics
ORDER BY Year

''');
data = pd.read_sql_query(query, conn)
data

,Year,Total_Sales,Total_Profit,Total_Orders,Previous_Year_Sales,YoY_Sales_Growth_Percent
0,2014,484247.50,49543.97,969,NaN,NaN
1,2015,470532.51,61618.60,1038,484247.50,-2.83
2,2016,609205.60,81795.17,1315,470532.51,29.47
3,2017,733215.26,93439.27,1687,609205.60,20.36


###**Moving Average Analysis**

In [123]:
query =('''
WITH MonthlySales AS (
SELECT
strftime('%Y', o.Order_Date) AS Year,
strftime('%m', o.Order_Date) AS Month,
ROUND(SUM(od.Sales), 2) AS Total_Sales,
ROUND(SUM(od.Profit), 2) AS Total_Profit,
COUNT(DISTINCT o.Order_ID) AS Total_Orders
FROM Orders o
JOIN Order_Details od
    ON o.Order_ID = od.Order_ID
GROUP BY Year, Month
ORDER BY Year, Month
),
MonthlyMetrics AS (
SELECT *,
LAG(Total_Sales) OVER (ORDER BY Year, Month) AS Previous_Month_Sales,
AVG(Total_Sales)
OVER (
    ORDER BY Year, Month
    ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
) AS Three_Month_Average_Sales
FROM MonthlySales
)
SELECT*,
ROUND(
    (Total_Sales - Previous_Month_Sales) * 100.0 /
    NULLIF(Previous_Month_Sales,0),
    2
) AS MoM_Sales_Growth_Percent
FROM MonthlyMetrics
ORDER BY Year, Month
''')
data = pd.read_sql_query(query, conn)
data

,Year,Month,Total_Sales,Total_Profit,Total_Orders,Previous_Month_Sales,Three_Month_Average_Sales,MoM_Sales_Growth_Percent
0,2014,01,14236.90,2450.19,32,NaN,14236.900000,NaN
1,2014,02,4519.89,862.31,28,14236.90,9378.395000,-68.25
2,2014,03,55691.01,498.73,71,4519.89,24815.933333,1132.13
3,2014,04,28295.35,3488.84,66,55691.01,29502.083333,-49.19
4,2014,05,23648.29,2738.71,69,28295.35,35878.216667,-16.42
5,2014,06,34595.13,4976.52,66,23648.29,28846.256667,46.29
6,2014,07,33946.39,-841.48,65,34595.13,30729.936667,-1.88
7,2014,08,27909.47,5318.11,72,33946.39,32150.330000,-17.78
8,2014,09,81777.35,8328.10,130,27909.47,47877.736667,193.01
9,2014,10,31453.39,3448.26,78,81777.35,47046.736667,-61.54


###**Seasonality Analysis**

In [129]:
query = ('''
WITH MonthlyData AS (
    SELECT
        strftime('%m', o.Order_Date) AS Month,
        ROUND(SUM(od.Sales), 2) AS Total_Sales,
        ROUND(AVG(od.Sales),2) AS Average_Order_Line_Sales,
        COUNT(DISTINCT o.Order_ID) AS Total_Orders
    FROM Orders o
    JOIN Order_Details od
        ON o.Order_ID = od.Order_ID
    GROUP BY Month
)
SELECT*
FROM MonthlyData
ORDER BY Month
''');
data = pd.read_sql_query(query, conn)
data

,Month,Total_Sales,Average_Order_Line_Sales,Total_Orders
0,01,94924.84,249.15,178
1,02,59751.25,199.17,162
2,03,205005.49,294.55,354
3,04,137762.13,206.23,343
4,05,155028.81,210.92,369
5,06,152718.68,213.00,364
6,07,147238.10,207.38,338
7,08,159044.06,225.27,341
8,09,307649.95,222.45,688
9,10,200322.98,244.59,417
